In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from uuid import uuid4



In [2]:
# Create a list of documents with metadata
docs = [
    Document(page_content="Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.",
             metadata={"topic": "space"}),
    Document(page_content="The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.",
             metadata={"topic": "space"}),
    Document(page_content="Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.",
             metadata={"topic": "space"}),
    Document(page_content="NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.",
             metadata={"topic": "space"}),
    Document(page_content="Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.",
             metadata={"topic": "space"}),
    Document(page_content="DNA is a double-helix molecule that carries the genetic instructions for all living organisms.",
             metadata={"topic": "biology"}),
    Document(page_content="Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.",
             metadata={"topic": "biology"}),
    Document(page_content="The Roman Empire at its peak covered over 5 million square kilometers across three continents.",
             metadata={"topic": "history"}),
    Document(page_content="The printing press, invented by Gutenberg around 1440, revolutionised the spread of knowledge.",
             metadata={"topic": "history"}),
    Document(page_content="The Amazon River discharges more freshwater into the ocean than any other river on Earth.",
             metadata={"topic": "geography"}),
    Document(page_content="The Sahara Desert spans about 9.2 million square kilometers across northern Africa.",
             metadata={"topic": "geography"}),
    Document(page_content="Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.",
             metadata={"topic": "biology"}),
]

In [3]:
for index,doc in enumerate(docs,1):
    print(f"DocNo. {index} | Topic: {doc.metadata["topic"]}")
    print(f"Content: {doc.page_content}\n")

DocNo. 1 | Topic: space
Content: Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

DocNo. 2 | Topic: space
Content: The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.

DocNo. 3 | Topic: space
Content: Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.

DocNo. 4 | Topic: space
Content: NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.

DocNo. 5 | Topic: space
Content: Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.

DocNo. 6 | Topic: biology
Content: DNA is a double-helix molecule that carries the genetic instructions for all living organisms.

DocNo. 7 | Topic: biology
Content: Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.

DocNo. 8 | Topic: history
Content: The Roman Empire at its peak covered over 5 million square kil

In [7]:
embeddings = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)


W1031 14:52:16.847000 18297 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9508.80it/s]


In [8]:
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="similarity-search-demo"
)

In [9]:
print(len(vector_store.get()["ids"]))

12


In [17]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [18]:
query = "How do cells generate their energy?"

# Retrievers inside langchain are runnables ,you can use them inside chains

results = retriever.invoke(query)

print(f"{query}\n")

for i,doc in enumerate(results,1):
    print(f"Result {i} [topic={doc.metadata['topic']}]")
    print(f"{doc.page_content}")
    print()

How do cells generate their energy?

Result 1 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 2 [topic=biology]
Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.

Result 3 [topic=space]
Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

Result 4 [topic=biology]
DNA is a double-helix molecule that carries the genetic instructions for all living organisms.

Result 5 [topic=space]
Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.



In [14]:
query = "How do cells generate their energy?"


# You do not use this method because the above method (i.e - Retriever method) gives you flexibility of using different search types and you can also tune search kwargs and also it is runnables so can be used inside chain which is not the case in this method.

results = vector_store.similarity_search(query,k=3)



print(f"{query}\n")

for i,doc in enumerate(results,1):
    print(f"Result {i} [topic={doc.metadata['topic']}]")
    print(f"{doc.page_content}")
    print()

How do cells generate their energy?

Result 1 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 2 [topic=biology]
Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.

Result 3 [topic=space]
Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

